# Arvinda Vema
# 114634757

# Projct 4

# Part 1: Part 1: Getting the Data
Installed the required packages, fetched the required data, and then cleaned the data up 

In [1]:
# Imports
import folium
import random
import matplotlib.pyplot as plt
import pandas
import json

# Reading Crime Data CVS into Pandas Dataframe
df = pandas.read_csv("https://cmsc320.github.io/files/BPD_Arrests.csv")

## Columns of interest
I am the most interested in seeing whether certain regions in Baltimore have historically been making more arrests compared to other regions within the city. 
Furthermore, I want to find out if a certain demographic is more likely to be arrested depending on which Police District they are from. Therefore I am only interested in the age, sex, race, location coordinates, and district column of the table. The rest of the columns are ignored.

In [2]:
# Handling values that arent valid
df = df[pandas.notnull(df["Location 1"])]

# Replacing nan values with "?" because it is important to indicate that data has gone missing after an arrest.
df['sex'] = df['sex'].fillna('?')
df['race'] = df['race'].fillna('?')

# Making latitude and longitude data more accessible
df["lat"], df["long"] = df["Location 1"].str.split(",").str
df["lat"] = df["lat"].str.replace("(", "", regex=False).astype(float)
df["long"] = df["long"].str.replace(")", "", regex = False).astype(float)

# dropping excess data columns
df = df.drop(columns=["Location 1", "arrestDate", "arrestTime", "incidentOffense", "arrestLocation", "incidentLocation", "charge", "chargeDescription", "post", "neighborhood"])

# Resulting DataFrame
df.head()

<ipython-input-2-4d92cabc12c8>:9: FutureWarning: Columnar iteration over characters will be deprecated in future releases.
  df["lat"], df["long"] = df["Location 1"].str.split(",").str


,arrest,age,race,sex,district,lat,long
1,11127013.0,37,B,M,SOUTHERN,39.281403,-76.648364
2,11126887.0,46,B,M,NORTHEASTERN,39.322770,-76.573575
3,11126873.0,50,B,M,WESTERN,39.311720,-76.662355
4,11126968.0,33,B,M,NORTHERN,39.338289,-76.604567
5,11127041.0,41,B,M,SOUTHERN,39.244989,-76.627358


# Part 2: Making the map
After reading the documentation for folium and plotting various types of geographical data, I implemented a map. It is centered on Baltimore and the base map was made to remove distractions such as roads and other irrelevant labels.

In [3]:
# Initializing a map "m" showing it
m = folium.Map(location=[39.29, -76.61],  tiles="cartodbpositron", zoom_start=12)
m

## Police Districts
To represent the district an arrest was made in, I was able to create a color coded layer for the base map which drew borders for each district. I obtained the GeoJson data from the official Baltimore City website at: https://data.baltimorecity.gov/datasets/956e52eb7abb4787abd7386e8efd600b_0?geometry=-76.902%2C39.192%2C-76.339%2C39.378. The layer was added as explained in the folium docs I read earlier.

In [4]:
# a set of unique colors that can each be mapped to one of the 9 Police Districts in Baltimore
# These districts are numbered 1 - 9 and also have a name. 
district_colors = [ 
    "blue",
    "green", 
    "orange", 
    "purple", 
    "yellow", 
    "red", 
    "pink", 
    "darkblue",
    "darkblue", 
    "black"
    ]

# This style function can be passed as an argument to the following operation to assign custom colors for each geographical object in the geojson file 
def style_function(feature):
    return {
        "color": district_colors[int(feature['properties']['district'])],
        "fillOpacity": 0.5,
        "weight": 1,
    }

# URL to GeoJson data 
balt_districts = 'https://opendata.arcgis.com/datasets/956e52eb7abb4787abd7386e8efd600b_0.geojson'

# Creating a layer for map m using the geojson data and the style function above. 
# Then added the layer to m
folium.GeoJson( balt_districts,  name="districts", style_function=style_function ).add_to(m)

# Result
m

## Race and Sex
The second and third variables I added to the markers was race and sex. Each race recieved a unique color. Each marker also has an icon indicating if the arrested was male, female, or missing. 

In [5]:
# Dictionary to translate the each race found in the data to a unique, and identifiable marker color.
race_colors = {
    'I': 'pink', 'W': 'lightgreen', 
    'A': 'darkred', 'U': 'lightblue', 
    'H': 'black','?': 'darkpurple',
    'B': 'orange'
    }

# Dictionary that translates sex into  icons from font awesome that clearly differentiate between male and female arrests.
# There is also a question mark third element which is meant to mark arrests with missing sex data
sex_icon = {
    'M': "male", 'F': "venus", 
    "?":"question-circle"
    }

## In order to demonstrate the results efficiently I have chosen to plot 100 randomly chosen observations from the data table

In [6]:
# Obtaining a list of 100 random numbers within the range of the data table
randoms = random.sample(range(0, 63892), 200)

# Process all 100 random rows
for i in randoms:
    row = df.iloc[i]
    # Initialize new marker and added it to map m
    folium.Marker( 
        [row['lat'], row['long']], 
        icon=folium.Icon(color=race_colors[str(row['race'])], icon=sex_icon[str(row['sex'])] , prefix="fa" ),
        ).add_to(m)

m

# Write-Up

## Racial Differences
A vast majority of the markers are always orange. Orange markers represent an arrest of a Black person. It is easy to jump to the conclusion that   

## Majority of Men
Men have been arrested significantly more women in Baltimore City. This pattern might be explained certain differences between men and women such as differences in development, and how each sex tends to cope with environmental triggers such as sociocultural stressors or financial instability. 

## District 5
During the coding process, I had to re run the code more than 20 times and each time I run the code, the map above takes a random sample where n = 200 out of the 60K+ observations in the data table and I am confounded by the way the markers on the map have never been placed within the red district. 

For an unknown reason, it appears that virtually 0 arrests occured within the boudries of the Northern district(District 5). Possible explanations might include:
### 1. Everyone in District 5 is very well behaved
Everyone abides the law and has never broken any laws.

### 2. District 5 is not very habitable 
The living conditions are not suitable for mankind. Therefore the population within District 5 is so low, that it is almost non-existant.

### 3. District 5 is actually a suburban area that was a created in the 1950's.
A vast majority of suburbs in the United states were created during this period in an attempt to abandon metropolitan. The need to escape was sparked when Black Americans began to move into city areas as they steadily climbed the socioeconomic ladder. Unfortunately, this resulted in the mass exodus of White Americans to the suburbs. The result was a vacuum of wealth in the cities, and a large population of Black Americans. It is not a secret that racism is real. Unfortunately, racism is so real that there is strong evidence in the form of arrest data. 